<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/AudioX-Turbo_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/AudioX-Turbo_Colab.ipynb" target="_parent"><img src="https://img.shields.io/badge/GitHub-View_Source-181717?logo=github" alt="View on GitHub"/></a>

# ⚡ AudioX-Turbo — 4-Step Anything-to-Audio, Wallpaper-to-SoundPack & Video Foley Studio

<p align="left">
  <a href="https://github.com/NoizAI/AudioX-Turbo"><img src="https://img.shields.io/badge/GitHub-NoizAI%2FAudioX--Turbo-181717?logo=github" alt="GitHub"></a>
  <a href="https://huggingface.co/HKUSTAudio/AudioX-Turbo"><img src="https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-HKUSTAudio%2FAudioX--Turbo-amber" alt="HuggingFace"></a>
  <img src="https://img.shields.io/badge/Speed-4--Step%20DMD%20Distilled%20(25x%20Faster)-brightgreen" alt="4-Step DMD">
  <img src="https://img.shields.io/badge/Modalities-Text%20%2B%20Image%20%2B%20Video%20%2B%20Audio-purple" alt="Multimodal">
</p>

Welcome to the **AudioX-Turbo** Google Colab studio! Developed by **HKUST** and **Noiz AI** ([`NoizAI/AudioX-Turbo`](https://github.com/NoizAI/AudioX-Turbo) / [`HKUSTAudio/AudioX-Turbo`](https://huggingface.co/HKUSTAudio/AudioX-Turbo)), `AudioX-Turbo` distills the Multimodal Adaptive Fusion (MAF) Diffusion Transformer into a **4-step Distribution Matching Distillation (DMD)** student model:

- **⚡ 4-Step CFG-Free Synthesis (~25× Faster NFE)** — Generates **44.1 kHz stereo** sound effects and music in just **4 function evaluations** without needing Classifier-Free Guidance.
- **🖼️ Wallpaper / Image + Text $\rightarrow$ Windows Sound Event Pack** — Upload your **desktop wallpaper image**! The studio converts your wallpaper into a visual conditioning sequence for `CLIP-ViT-B/32` + `Synchformer` so your generated **Windows 10/11 Sound Scheme** matches the visual mood and aesthetic of your desktop theme!
- **🎬 Synchronized Video-to-Audio (V2A / TV2A) Foley** — Upload any silent video or pick directly from your Google Drive `/content/drive/MyDrive/` video exports (`Wan2.2`, `Wan-VACE`, `FramePack`, `SeedVR2`) and synthesize synchronized Foley + auto-muxed `.mp4` in 4 steps.
- **🔄 Audio + Text Conditioning (TA2A)** — Condition on a reference `.wav` clip + text prompt to create fast acoustic variations.

In [ ]:
#@title Step 1: Verify GPU & Install AudioX-Turbo Dependencies
import os
import sys
import subprocess
import torch
from tqdm.auto import tqdm
from IPython.display import clear_output

REPO_DIR = "/content/AudioX-Turbo"

def run_cmd(cmd, check=False):
    print(f"⏳ Running: {cmd}")
    return subprocess.run(cmd, shell=True, check=check)

try:
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"✅ GPU Detected: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    else:
        print("⚠️ No CUDA GPU detected — GPU is strongly recommended.")

    if not os.path.exists(REPO_DIR):
        run_cmd(f"git clone --depth 1 https://github.com/NoizAI/AudioX-Turbo.git {REPO_DIR}", check=True)

    run_cmd("apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1", check=False)

    pip_pkgs = (
        "einops einops-exts safetensors soundfile>=0.12.1 huggingface_hub>=0.26.0 "
        "transformers>=4.46.0 accelerate omegaconf pytorch-lightning "
        "alias-free-torch descript-audio-codec vector-quantize-pytorch "
        "open-clip-torch laion-clap k-diffusion scipy librosa decord av moviepy "
        "gradio>=5.0.0 matplotlib timm torchvision torchaudio tqdm"
    )
    run_cmd(f"{sys.executable} -m pip install -q -U {pip_pkgs}", check=False)
    run_cmd(f"{sys.executable} -m pip install -q --no-deps -e {REPO_DIR}", check=False)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    clear_output()
    print("✅ Step 1 Complete: AudioX-Turbo environment installed!")
except Exception as e:
    print(f"❌ Installation Error: {e}")
    raise

In [ ]:
#@title Step 2: Mount Google Drive & Configure Persistent Cache
import os
import sys

WORKSPACE_DIR = "/content/audiox_turbo_ws"
os.makedirs(WORKSPACE_DIR, exist_ok=True)

DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = os.path.ismount("/content/drive")
except Exception as e:
    print(f"ℹ️ Google Drive not mounted: {e}")

if DRIVE_MOUNTED:
    CACHE_ROOT = "/content/drive/MyDrive/AEI_Audio_Cache/audiox_turbo"
    OUTPUT_DIR = "/content/drive/MyDrive/AEI_Audio_Exports/AudioX_Turbo"
else:
    CACHE_ROOT = os.path.join(WORKSPACE_DIR, "cache")
    OUTPUT_DIR = os.path.join(WORKSPACE_DIR, "outputs")

for d in (CACHE_ROOT, OUTPUT_DIR):
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = os.path.join(CACHE_ROOT, "huggingface")
os.environ["TORCH_HOME"] = os.path.join(CACHE_ROOT, "torch")
os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

print(f"📂 HF_HOME Cache : {os.environ['HF_HOME']}")
print(f"📂 Output Exports: {OUTPUT_DIR}")

In [ ]:
#@title Step 3: Core AudioX-Turbo 4-Step DMD Engine, Wallpaper Video Helper & Packager
import os
import gc
import sys
import time
import glob
import subprocess
import torch
import torchaudio
import numpy as np
import soundfile as sf
from PIL import Image
from einops import rearrange
from huggingface_hub import hf_hub_download

REPO_DIR = "/content/AudioX-Turbo"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ==============================================================================
# Shared Windows 10/11 Sound Scheme & Zero-Latency UI Mastering Engine
# ==============================================================================
import io
import re
import json
import uuid
import base64
import shutil
import zipfile
from datetime import datetime
import scipy.signal
import numpy as np
import soundfile as sf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

WINDOWS_SYSTEM_EVENTS = {
    "WindowsLogon": {
        "label": "Windows Logon / Startup",
        "reg_app": ".Default",
        "duration": 3.0,
        "contour": "warm welcoming ascending chord swell with a gentle shimmering reverb tail",
    },
    "WindowsLogoff": {
        "label": "Windows Logoff / Shutdown",
        "reg_app": ".Default",
        "duration": 2.5,
        "contour": "peaceful resolving descending harmonic chord that fades smoothly into silence",
    },
    "WindowsUnlock": {
        "label": "Windows Unlock / Sign-In",
        "reg_app": ".Default",
        "duration": 1.4,
        "contour": "bright welcoming two-note ascending unlock chime with airy resonance",
    },
    "SystemNotification": {
        "label": "System Notification (Toast)",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "crisp pleasant two-note notification chime, clear transient attack, short clean tail",
    },
    "WindowsNotifySystemGeneric": {
        "label": "Action Center / Generic Toast Banner",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "modern crystalline two-tone notification ping with soft harmonic decay",
    },
    "SystemAsterisk": {
        "label": "Asterisk / Info Pop-up",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "soft single crystalline ping with subtle harmonic resonance",
    },
    "SystemExclamation": {
        "label": "Exclamation / Warning Alert",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "attention-grabbing mid-range double pulse chime, alert but non-harsh",
    },
    "SystemHand": {
        "label": "Critical Stop / Error",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "low muted percussive double thud with a tense minor interval, error alert",
    },
    "SystemQuestion": {
        "label": "Question / Confirmation Dialog",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "gentle inquisitive rising two-note harmonic inflection chime",
    },
    "Default": {
        "label": "Default Beep / Fallback Alert",
        "reg_app": ".Default",
        "duration": 0.8,
        "contour": "clean short balanced harmonic ping, neutral interface alert",
    },
    "DeviceConnect": {
        "label": "Device Connect (USB Plug-In)",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "upbeat two-tone ascending interval (low note to higher fifth), hardware connected",
    },
    "DeviceDisconnect": {
        "label": "Device Disconnect (USB Unplug)",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "smooth two-tone descending interval (high note to lower root), hardware unplugged",
    },
    "DeviceFail": {
        "label": "Device Failed to Connect",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "short stuttering muted low buzz double tap, hardware connection failure",
    },
    "EmptyRecycleBin": {
        "label": "Empty Recycle Bin",
        "reg_app": "Explorer",
        "duration": 1.2,
        "contour": "crisp satisfying paper and glass shimmer dissolve whoosh",
    },
    "WindowsUAC": {
        "label": "User Account Control (UAC Prompt)",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "authoritative clean two-note security confirmation chime",
    },
    "LowBatteryAlarm": {
        "label": "Low Battery Alarm",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "gentle pulsing two-note reminder tone",
    },
    "CriticalBatteryAlarm": {
        "label": "Critical Battery Alarm",
        "reg_app": ".Default",
        "duration": 1.5,
        "contour": "urgent rapid triple pulse warning chime",
    },
    "MailBeep": {
        "label": "New Mail / Message Beep",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "bright airy bell ding notification",
    },
    "MessageNudge": {
        "label": "Message Nudge / Chat Ping",
        "reg_app": ".Default",
        "duration": 0.9,
        "contour": "playful double tactile pop ping notification",
    },
    "PrintComplete": {
        "label": "Print / Task Complete",
        "reg_app": ".Default",
        "duration": 1.3,
        "contour": "satisfying upbeat three-note completion flourish chime",
    },
    "MenuCommand": {
        "label": "Menu Command / Tactile UI Click",
        "reg_app": ".Default",
        "duration": 0.6,
        "contour": "ultra-short crisp tactile UI click pop transient, dry and immediate",
    },
}

SONIC_THEME_PRESETS = {
    "Frutiger Aero Crystal Glass": (
        "crystalline water-drop glass marimba and airy harmonic bell pads, "
        "clean high-tech Frutiger Aero operating system UI sound effect, studio quality"
    ),
    "Cyberpunk Neon HUD": (
        "futuristic cyberpunk holographic HUD telemetry synthesizer, "
        "crisp digital micro-transient with warm sub-bass resonance, sci-fi UI sound"
    ),
    "Cozy Wooden Kalimba & Mallet": (
        "warm organic rosewood marimba, thumb piano kalimba, and soft felt mallet percussion, "
        "intimate close-mic acoustic UI sound effect"
    ),
    "Sci-Fi Spaceship Bridge": (
        "clean starship computer console telemetry bleep, precision FM synthesis harmonic tone, "
        "futuristic spacecraft interface sound effect"
    ),
    "Retro 16-Bit Chiptune Arcade": (
        "crisp nostalgic 16-bit FM synth arcade game UI sound effect, "
        "bright square and triangle wave arpeggio, clean retro console audio"
    ),
    "Warm Lo-Fi Analog Tape Synth": (
        "warm vintage analog poly-synth and Rhodes electric piano chime with gentle tape saturation, "
        "cozy lo-fi interface sound effect"
    ),
    "Minimalist Tactile Micro-Foley": (
        "ultra-clean modern minimalist UI micro-foley, soft mechanical switch click and muted glass tap, "
        "tight dry studio recording"
    ),
    "Dark Fantasy Crystal & Harp": (
        "ethereal celesta, delicate harp harmonic pluck, and mystical resonant crystal chime, "
        "fantasy RPG interface sound effect"
    ),
}

DEFAULT_SELECTED_WIN_EVENTS = [
    "WindowsLogon",
    "WindowsLogoff",
    "WindowsUnlock",
    "SystemNotification",
    "WindowsNotifySystemGeneric",
    "SystemAsterisk",
    "SystemExclamation",
    "SystemHand",
    "DeviceConnect",
    "DeviceDisconnect",
    "EmptyRecycleBin",
    "WindowsUAC",
]


def build_windows_event_prompt(event_key: str, theme_name: str, custom_style: str = "") -> str:
    ev = WINDOWS_SYSTEM_EVENTS.get(event_key, WINDOWS_SYSTEM_EVENTS["SystemNotification"])
    style_desc = custom_style.strip() if custom_style and custom_style.strip() else SONIC_THEME_PRESETS.get(
        theme_name, SONIC_THEME_PRESETS["Frutiger Aero Crystal Glass"]
    )
    return f"{ev['contour']}, {style_desc}, high fidelity, no background noise"


def apply_phase_safe_stereo_widener(arr: np.ndarray, sr: int, width: float = 0.22) -> np.ndarray:
    """Upgrade mono (N, 1) to phase-coherent stereo (N, 2) or subtly widen stereo above 350 Hz."""
    if arr.ndim == 1:
        arr = arr[:, None]
    n_samples, ch = arr.shape
    if n_samples < 64 or width <= 0.01:
        return np.repeat(arr, 2, axis=1) if ch == 1 else arr

    nyq = 0.5 * float(sr)
    cutoff = min(max(350.0 / nyq, 0.01), 0.85)
    b, a = scipy.signal.butter(2, cutoff, btype="highpass")

    if ch == 1:
        mono = arr[:, 0]
        high = scipy.signal.lfilter(b, a, mono).astype(np.float32)
        delay_samples = max(1, int(0.00038 * sr))  # ~0.38 ms Haas micro-decorrelation
        side = np.zeros_like(high)
        if delay_samples < n_samples:
            side[delay_samples:] = high[:-delay_samples] - high[delay_samples:]
        side *= float(width)
        left = mono + side
        right = mono - side
        return np.stack([left, right], axis=1).astype(np.float32)
    else:
        mid = 0.5 * (arr[:, 0] + arr[:, 1])
        side = 0.5 * (arr[:, 0] - arr[:, 1])
        side_high = scipy.signal.lfilter(b, a, side).astype(np.float32)
        side_boosted = side + float(width) * 0.5 * side_high
        return np.stack([mid + side_boosted, mid - side_boosted], axis=1).astype(np.float32)


def master_ui_sound(
    audio_np: np.ndarray,
    sr: int,
    target_duration_s: float = None,
    trim_silence: bool = True,
    trim_db: float = -45.0,
    fade_in_ms: float = 1.5,
    fade_out_ms: float = 120.0,
    target_dbfs: float = -1.2,
    stereo_widen: bool = True,
    hpf_hz: float = 40.0,
    pitch_semitones: float = 0.0,
) -> np.ndarray:
    """Zero-latency UI sound mastering: snaps to first transient (<1ms pre-roll), 40Hz sub-rumble filter, optional semitone transpose, 1.5ms click-free micro-fade, phase-safe stereo widening, and true-peak normalization."""
    arr = np.asarray(audio_np, dtype=np.float32)
    if arr.ndim == 1:
        arr = arr[:, None]  # (samples, 1)
    elif arr.ndim == 2 and arr.shape[0] <= 2 and arr.shape[1] > arr.shape[0]:
        arr = arr.T  # (samples, channels)

    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr = arr - np.mean(arr, axis=0, keepdims=True)

    # Optional sub-rumble high-pass filter (default 40 Hz) to keep UI chimes crisp
    if hpf_hz and float(hpf_hz) > 10.0 and arr.shape[0] > 64:
        nyq = 0.5 * float(sr)
        cutoff = min(max(float(hpf_hz) / nyq, 0.002), 0.45)
        b_hp, a_hp = scipy.signal.butter(2, cutoff, btype="highpass")
        arr = scipy.signal.lfilter(b_hp, a_hp, arr, axis=0).astype(np.float32)

    # Optional semitone pitch shift via high-precision polyphase resampling
    if pitch_semitones and abs(float(pitch_semitones)) >= 0.05 and arr.shape[0] > 64:
        factor = 2.0 ** (float(pitch_semitones) / 12.0)
        up = 1000
        down = max(100, int(round(1000.0 * factor)))
        gcd = np.gcd(up, down)
        arr = scipy.signal.resample_poly(arr, up // gcd, down // gcd, axis=0).astype(np.float32)

    if trim_silence and arr.shape[0] > int(0.04 * sr):
        mono_env = np.max(np.abs(arr), axis=1)
        diff_env = np.abs(np.diff(mono_env, prepend=mono_env[0]))
        peak = float(np.max(mono_env))
        if peak > 1e-6:
            amp_thresh = peak * (10.0 ** (float(trim_db) / 20.0))
            flux_thresh = float(np.max(diff_env)) * 0.08
            active_idx = np.where((mono_env >= amp_thresh) | (diff_env >= flux_thresh))[0]
            if active_idx.size > 0:
                pre_roll = int(0.0008 * sr)   # 0.8 ms zero-latency pre-roll
                post_roll = int(0.160 * sr)   # 160 ms natural reverb tail retention
                start_i = max(0, int(active_idx[0]) - pre_roll)
                end_i = min(arr.shape[0], int(active_idx[-1]) + post_roll)
                if end_i > start_i + int(0.03 * sr):
                    arr = arr[start_i:end_i]

    if target_duration_s is not None and target_duration_s > 0:
        max_samples = int(float(target_duration_s) * sr)
        if arr.shape[0] > max_samples:
            arr = arr[:max_samples]

    n_samples = arr.shape[0]
    if n_samples == 0:
        return np.zeros((int(0.5 * sr), 2), dtype=np.float32)

    if stereo_widen:
        arr = apply_phase_safe_stereo_widener(arr, sr, width=0.22 if arr.shape[1] == 1 else 0.12)

    fi_samples = min(int((float(fade_in_ms) / 1000.0) * sr), max(1, n_samples // 8))
    if fi_samples > 1:
        fi_curve = 0.5 * (1.0 - np.cos(np.linspace(0.0, np.pi, fi_samples, dtype=np.float32)))
        arr[:fi_samples] *= fi_curve[:, None]

    fo_samples = min(int((float(fade_out_ms) / 1000.0) * sr), max(1, n_samples // 2))
    if fo_samples > 1:
        fo_curve = (0.5 * (1.0 + np.cos(np.linspace(0.0, np.pi, fo_samples, dtype=np.float32)))) ** 1.2
        arr[-fo_samples:] *= fo_curve[:, None]

    peak_val = float(np.max(np.abs(arr)))
    if peak_val > 1e-6:
        target_amp = 10.0 ** (float(target_dbfs) / 20.0)
        arr = arr * (target_amp / peak_val)

    arr = np.clip(arr, -0.999, 0.999)
    return arr if arr.shape[1] > 1 else arr[:, 0]


def save_pcm16_wav(path: str, audio_np: np.ndarray, sr: int, subtype: str = "PCM_16") -> str:
    abs_path = os.path.abspath(path)
    os.makedirs(os.path.dirname(abs_path), exist_ok=True)
    sf.write(abs_path, audio_np, int(sr), subtype=subtype)
    # Mirror to AEI_TTS_Out if Google Drive is mounted so companion notebooks (InfiniteTalk, SeedVR2, Audio_PostProcessor) auto-discover it
    if "/content/drive/MyDrive/" in abs_path and "/AEI_TTS_Out/" not in abs_path:
        try:
            mirror_dir = "/content/drive/MyDrive/AEI_TTS_Out/SFX_Exports"
            os.makedirs(mirror_dir, exist_ok=True)
            shutil.copy2(abs_path, os.path.join(mirror_dir, os.path.basename(abs_path)))
        except Exception:
            pass
    return abs_path


def load_audio_file_safe(audio_path: str, target_sr: int = None, mono: bool = False):
    """Robust audio loader using soundfile first, falling back to librosa/torchaudio."""
    try:
        data, sr = sf.read(audio_path, dtype="float32", always_2d=True)
        # data is (T, C) -> transpose to (C, T)
        wav = data.T
    except Exception:
        import librosa
        wav, sr = librosa.load(audio_path, sr=None, mono=False)
        if wav.ndim == 1:
            wav = wav[None, :]
    if mono and wav.shape[0] > 1:
        wav = np.mean(wav, axis=0, keepdims=True)
    if target_sr is not None and int(sr) != int(target_sr):
        gcd = np.gcd(int(sr), int(target_sr))
        wav = scipy.signal.resample_poly(wav, int(target_sr) // gcd, int(sr) // gcd, axis=-1).astype(np.float32)
        sr = int(target_sr)
    return wav, int(sr)


def render_waveform_and_spectrogram(wav_path: str, title: str = "Audio Signal & Spectrogram") -> str:
    """Render a dual-panel waveform + spectrogram plot for visual transient & tail inspection."""
    try:
        data, sr = sf.read(wav_path, always_2d=True)
        mono = np.mean(data, axis=1)
        dur = len(mono) / float(sr)
        t = np.linspace(0.0, dur, len(mono))

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8.5, 4.2), facecolor="#0f172a")
        for ax in (ax1, ax2):
            ax.set_facecolor("#1e293b")
            ax.tick_params(colors="#cbd5e1", labelsize=8)
            for spine in ax.spines.values():
                spine.set_color("#334155")

        if data.shape[1] > 1:
            ax1.plot(t, data[:, 0], color="#38bdf8", linewidth=0.8, alpha=0.85, label="L")
            ax1.plot(t, data[:, 1], color="#f472b6", linewidth=0.8, alpha=0.75, label="R")
            ax1.legend(loc="upper right", fontsize=7, facecolor="#0f172a", edgecolor="#334155", labelcolor="#f8fafc")
        else:
            ax1.plot(t, mono, color="#38bdf8", linewidth=0.9)
        ax1.set_xlim(0, max(dur, 0.05))
        ax1.set_ylim(-1.05, 1.05)
        ax1.set_ylabel("Amplitude", color="#e2e8f0", fontsize=8)
        ch_label = "Stereo" if data.shape[1] > 1 else "Mono"
        ax1.set_title(f"{title} ({sr} Hz {ch_label}, {dur:.2f}s)", color="#f8fafc", fontsize=10, fontweight="bold")
        ax1.grid(True, color="#334155", alpha=0.4, linestyle="--")

        nfft = min(1024, max(128, 2 ** int(np.log2(max(len(mono) // 8, 128)))))
        ax2.specgram(mono + 1e-10, NFFT=nfft, Fs=sr, noverlap=nfft // 2, cmap="magma")
        ax2.set_xlim(0, max(dur, 0.05))
        ax2.set_xlabel("Time (seconds)", color="#e2e8f0", fontsize=8)
        ax2.set_ylabel("Freq (Hz)", color="#e2e8f0", fontsize=8)

        plt.tight_layout()
        out_png = os.path.join("/tmp", f"spec_{uuid.uuid4().hex[:8]}.png")
        fig.savefig(out_png, dpi=120, facecolor=fig.get_facecolor())
        plt.close(fig)
        return out_png
    except Exception as e:
        print(f"Spectrogram render warning: {e}")
        return None


def build_audition_soundboard_html(
    scheme_name: str,
    event_wav_map: dict,
    theme_name: str,
    model_label: str,
    embed_base64: bool = False,
) -> str:
    """Build a dark-mode HTML5 WebAudio interactive soundboard with 1-click sequential Play-All auditioning."""
    board_uid = f"sb_{uuid.uuid4().hex[:8]}"
    cards_html = []
    for ev_key, wav_info in event_wav_map.items():
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, {"label": ev_key, "reg_app": ".Default", "duration": 1.0})
        if embed_base64 and os.path.exists(wav_info["path"]):
            with open(wav_info["path"], "rb") as bf:
                b64 = base64.b64encode(bf.read()).decode("ascii")
            audio_src = f"data:audio/wav;base64,{b64}"
        else:
            audio_src = f"Media/{wav_info['filename']}"

        cards_html.append(
            f'<div class="aei-sb-card" style="background:#1e293b;border:1px solid #334155;border-radius:10px;padding:10px 12px;display:flex;flex-direction:column;gap:6px;transition:all 0.2s ease;">'
            f'<div style="display:flex;justify-content:space-between;align-items:center;">'
            f'<strong style="color:#f8fafc;font-size:13px;">🔔 {ev_meta.get("label", ev_key)}</strong>'
            f'<span style="background:#0f172a;color:#38bdf8;font-size:11px;padding:2px 6px;border-radius:4px;font-family:monospace;">{ev_meta.get("reg_app", ".Default")}\\{ev_key}</span>'
            f'</div>'
            f'<audio controls preload="none" src="{audio_src}" style="width:100%;height:32px;"></audio>'
            f'</div>'
        )
    grid_str = "\n".join(cards_html)
    play_all_js = (
        f"const root=document.getElementById('{board_uid}');"
        f"if(!root)return;"
        f"const audios=Array.from(root.querySelectorAll('audio'));"
        f"const cards=Array.from(root.querySelectorAll('.aei-sb-card'));"
        f"if(root._playing){{root._playing=false;audios.forEach(a=>{{a.pause();a.currentTime=0;}});cards.forEach(c=>c.style.borderColor='#334155');this.textContent='▶ Play All Events Sequentially';return;}}"
        f"root._playing=true;this.textContent='⏹ Stop Sequence';"
        f"let idx=0;const btn=this;"
        f"const next=()=>{{if(!root._playing||idx>=audios.length){{root._playing=false;btn.textContent='▶ Play All Events Sequentially';cards.forEach(c=>c.style.borderColor='#334155');return;}}"
        f"cards.forEach((c,i)=>{{c.style.borderColor=(i===idx)?'#38bdf8':'#334155';}});"
        f"const a=audios[idx++];a.currentTime=0;"
        f"a.onended=()=>{{setTimeout(next,280);}};"
        f"a.onerror=()=>{{setTimeout(next,100);}};"
        f"a.play().catch(()=>setTimeout(next,100));}};"
        f"next();"
    )
    return (
        f'<div id="{board_uid}" style="background:#0f172a;color:#f8fafc;padding:16px;border-radius:12px;border:1px solid #334155;font-family:system-ui,-apple-system,sans-serif;">'
        f'<div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:10px;margin-bottom:12px;">'
        f'<div>'
        f'<h3 style="margin:0 0 4px 0;color:#38bdf8;font-size:16px;">🪟 {scheme_name} — Interactive Windows 10/11 Soundboard</h3>'
        f'<p style="margin:0;color:#94a3b8;font-size:12px;">Theme: <b>{theme_name}</b> · Engine: <b>{model_label}</b> · Events: <b>{len(event_wav_map)}</b></p>'
        f'</div>'
        f'<button type="button" onclick="{play_all_js}" style="background:#0284c7;color:#ffffff;border:none;border-radius:8px;padding:8px 14px;font-size:12px;font-weight:600;cursor:pointer;">▶ Play All Events Sequentially</button>'
        f'</div>'
        f'<div style="display:grid;grid-template-columns:repeat(auto-fill,minmax(270px,1fr));gap:10px;">'
        f'{grid_str}'
        f'</div>'
        f'</div>'
    )


LAST_WINPACK_STATE = {
    "pack_dir": None,
    "scheme_name": "AEI_Custom_Scheme",
    "theme_name": "Frutiger Aero Crystal Glass",
    "model_label": "AEI Sound Studio",
    "output_dir": "/content",
    "event_wav_map": {},
}


def package_windows_sound_scheme_zip(
    scheme_name: str,
    event_wav_map: dict,
    theme_name: str,
    model_label: str,
    output_dir: str,
) -> str:
    """Create a complete Windows 10/11 Sound Scheme ZIP with .wav files, Install/Uninstall .ps1, .reg file, Scheme.ini, soundpack_manifest.json, and audition_board.html."""
    safe_id = re.sub(r"[^A-Za-z0-9_-]+", "_", (scheme_name or "AEI_Custom_Sounds").strip())[:32].strip("_") or "AEI_Custom"
    display_name = (scheme_name or "AEI Custom Sounds").strip()[:48]
    os.makedirs(output_dir, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_path = os.path.join(output_dir, f"{safe_id}_Windows_SoundPack_{ts}.zip")

    ps1_lines = [
        f"# ==============================================================================",
        f"# Windows 10/11 Sound Scheme Automated Installer: {display_name}",
        f"# Generated by AEI Colab Notebooks ({model_label})",
        f"# ==============================================================================",
        f"$ErrorActionPreference = 'Stop'",
        f"$SchemeId = '{safe_id}'",
        f"$SchemeDisplay = '{display_name}'",
        f"$TargetDir = Join-Path $env:LOCALAPPDATA \"WindowsSoundSchemes\\$SchemeId\"",
        f"New-Item -ItemType Directory -Force -Path $TargetDir | Out-Null",
        f"$BackupReg = Join-Path $TargetDir 'Backup_Previous_AppEvents.reg'",
        f"if (-not (Test-Path $BackupReg)) {{ reg export 'HKCU\\AppEvents\\Schemes' $BackupReg /y | Out-Null }}",
        f"Copy-Item -Path (Join-Path $PSScriptRoot 'Media\\*.wav') -Destination $TargetDir -Force",
        f"",
        f"$NamesKey = \"HKCU:\\AppEvents\\Schemes\\Names\\$SchemeId\"",
        f"New-Item -Path $NamesKey -Force | Out-Null",
        f"Set-ItemProperty -Path $NamesKey -Name '(Default)' -Value $SchemeDisplay",
        f"",
        f"$EventMap = @{{",
    ]
    for ev_key, wav_info in event_wav_map.items():
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, {"reg_app": ".Default"})
        ps1_lines.append(f"    '{ev_meta['reg_app']}\\{ev_key}' = '{wav_info['filename']}'")
    ps1_lines += [
        f"}}",
        f"$FirstWav = $null",
        f"foreach ($entry in $EventMap.GetEnumerator()) {{",
        f"    $wavPath = Join-Path $TargetDir $entry.Value",
        f"    if (-not $FirstWav -and (Test-Path $wavPath)) {{ $FirstWav = $wavPath }}",
        f"    $schemeEventKey = \"HKCU:\\AppEvents\\Schemes\\Apps\\$($entry.Key)\\$SchemeId\"",
        f"    $currentEventKey = \"HKCU:\\AppEvents\\Schemes\\Apps\\$($entry.Key)\\.Current\"",
        f"    New-Item -Path $schemeEventKey -Force | Out-Null",
        f"    Set-ItemProperty -Path $schemeEventKey -Name '(Default)' -Value $wavPath",
        f"    New-Item -Path $currentEventKey -Force | Out-Null",
        f"    Set-ItemProperty -Path $currentEventKey -Name '(Default)' -Value $wavPath",
        f"}}",
        f"Set-ItemProperty -Path 'HKCU:\\AppEvents\\Schemes' -Name '(Default)' -Value $SchemeId",
        f"if ($FirstWav) {{ try {{ (New-Object System.Media.SoundPlayer $FirstWav).Play() }} catch {{}} }}",
        f"Write-Host \"Successfully installed and activated Windows Sound Scheme: $SchemeDisplay\" -ForegroundColor Green",
        f"Write-Host \"Prior registry state backed up to: $BackupReg\" -ForegroundColor Cyan",
        f"Start-Process 'control.exe' -ArgumentList 'mmsys.cpl,,2'",
    ]

    uninstall_ps1 = [
        f"# ==============================================================================",
        f"# Windows 10/11 Sound Scheme Uninstaller & Default Restore: {display_name}",
        f"# ==============================================================================",
        f"$ErrorActionPreference = 'SilentlyContinue'",
        f"$SchemeId = '{safe_id}'",
        f"Set-ItemProperty -Path 'HKCU:\\AppEvents\\Schemes' -Name '(Default)' -Value '.Default'",
        f"Get-ChildItem 'HKCU:\\AppEvents\\Schemes\\Apps' -Recurse | Where-Object {{ $_.PSChildName -eq $SchemeId }} | Remove-Item -Recurse -Force",
        f"Remove-Item -Path \"HKCU:\\AppEvents\\Schemes\\Names\\$SchemeId\" -Recurse -Force",
        f"$TargetDir = Join-Path $env:LOCALAPPDATA \"WindowsSoundSchemes\\$SchemeId\"",
        f"if (Test-Path $TargetDir) {{ Remove-Item -Path $TargetDir -Recurse -Force }}",
        f"Write-Host \"Restored Windows Default sound scheme and removed $SchemeId.\" -ForegroundColor Yellow",
        f"Start-Process 'control.exe' -ArgumentList 'mmsys.cpl,,2'",
    ]

    reg_lines = [
        "Windows Registry Editor Version 5.00",
        "",
        f"; Windows Sound Scheme: {display_name} ({model_label})",
        f"[HKEY_CURRENT_USER\\AppEvents\\Schemes\\Names\\{safe_id}]",
        f"@=\"{display_name}\"",
        "",
    ]
    for ev_key, wav_info in event_wav_map.items():
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, {"reg_app": ".Default"})
        escaped_path = f"C:\\\\Windows\\\\Media\\\\{safe_id}\\\\{wav_info['filename']}"
        reg_lines += [
            f"[HKEY_CURRENT_USER\\AppEvents\\Schemes\\Apps\\{ev_meta['reg_app']}\\{ev_key}\\{safe_id}]",
            f"@=\"{escaped_path}\"",
            "",
            f"[HKEY_CURRENT_USER\\AppEvents\\Schemes\\Apps\\{ev_meta['reg_app']}\\{ev_key}\\.Current]",
            f"@=\"{escaped_path}\"",
            "",
        ]

    ini_lines = [
        "[Scheme]",
        f"Name={display_name}",
        f"SchemeId={safe_id}",
        f"Theme={theme_name}",
        f"Generator={model_label}",
        f"CreatedAt={datetime.now().isoformat()}",
        "",
        "[Events]",
    ]
    for ev_key, wav_info in event_wav_map.items():
        ini_lines.append(f"{ev_key}=Media/{wav_info['filename']}")
        ini_lines.append(f"{ev_key}_Prompt={wav_info.get('prompt', '')}")

    manifest_data = {
        "scheme_name": display_name,
        "scheme_id": safe_id,
        "theme_name": theme_name,
        "generator": model_label,
        "created_at": datetime.now().isoformat(),
        "events": {
            k: {
                "filename": f"Media/{v['filename']}",
                "label": WINDOWS_SYSTEM_EVENTS.get(k, {}).get("label", k),
                "reg_app": WINDOWS_SYSTEM_EVENTS.get(k, {}).get("reg_app", ".Default"),
                "prompt": v.get("prompt", ""),
            }
            for k, v in event_wav_map.items()
        },
    }

    standalone_html = (
        "<!DOCTYPE html>\n<html><head><meta charset='utf-8'>"
        f"<title>{display_name} — Windows Soundboard</title></head>"
        "<body style='background:#020617;margin:24px;'>"
        + build_audition_soundboard_html(display_name, event_wav_map, theme_name, model_label, embed_base64=False)
        + "</body></html>"
    )

    readme_txt = (
        f"====================================================================\n"
        f"  {display_name} — Windows 10 / 11 Sound Event Pack\n"
        f"  Generated with: {model_label}\n"
        f"  Theme Preset:   {theme_name}\n"
        f"====================================================================\n\n"
        f"PREVIEW IN BROWSER FIRST:\n"
        f"  Open 'audition_board.html' in any browser to audition all {len(event_wav_map)} events!\n\n"
        f"OPTION 1: 1-Click PowerShell Install (Recommended, No Admin Required)\n"
        f"--------------------------------------------------------------------\n"
        f"1. Extract this entire ZIP archive to any folder.\n"
        f"2. Right-click 'Install_{safe_id}.ps1' -> 'Run with PowerShell'.\n"
        f"3. The script copies the .wav files to %LOCALAPPDATA%\\WindowsSoundSchemes\\{safe_id},\n"
        f"   registers '{display_name}' in the Windows Sound Control Panel, activates it,\n"
        f"   and opens mmsys.cpl so you can preview or customize individual events!\n"
        f"4. To revert at any time, right-click 'Uninstall_{safe_id}.ps1' -> 'Run with PowerShell'.\n\n"
        f"OPTION 2: Manual Registry (.reg) Import\n"
        f"--------------------------------------------------------------------\n"
        f"1. Copy the 'Media' folder contents into 'C:\\Windows\\Media\\{safe_id}\\'.\n"
        f"2. Double-click 'Install_{safe_id}.reg' to register the sound scheme.\n"
    )

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for ev_key, wav_info in event_wav_map.items():
            zf.write(wav_info["path"], arcname=f"Media/{wav_info['filename']}")
        zf.writestr(f"Install_{safe_id}.ps1", "\r\n".join(ps1_lines))
        zf.writestr(f"Uninstall_{safe_id}.ps1", "\r\n".join(uninstall_ps1))
        zf.writestr(f"Install_{safe_id}.reg", "\r\n".join(reg_lines))
        zf.writestr("Scheme.ini", "\r\n".join(ini_lines))
        zf.writestr("soundpack_manifest.json", json.dumps(manifest_data, indent=2))
        zf.writestr("audition_board.html", standalone_html)
        zf.writestr("README_INSTALL.txt", readme_txt.replace("\n", "\r\n"))

    if event_wav_map:
        first_entry = next(iter(event_wav_map.values()))
        if isinstance(first_entry, dict) and "path" in first_entry:
            LAST_WINPACK_STATE["pack_dir"] = os.path.dirname(first_entry["path"])
    LAST_WINPACK_STATE["scheme_name"] = display_name
    LAST_WINPACK_STATE["theme_name"] = theme_name
    LAST_WINPACK_STATE["model_label"] = model_label
    LAST_WINPACK_STATE["output_dir"] = output_dir
    LAST_WINPACK_STATE["event_wav_map"] = dict(event_wav_map)
    return zip_path


def finalize_rerolled_event_in_pack(
    ev_key: str,
    mastered_wav: np.ndarray,
    sr: int,
    ev_prompt: str,
    output_dir: str,
    scheme_name: str = "",
    theme_name: str = "",
    model_label: str = "",
):
    """Save a re-rolled single Windows event .wav into the active pack state, re-bundle the ZIP, and refresh the soundboard."""
    pack_dir = LAST_WINPACK_STATE.get("pack_dir")
    if not pack_dir or not os.path.exists(pack_dir):
        pack_dir = os.path.join(output_dir, f"winpack_reroll_{int(datetime.now().timestamp())}")
        os.makedirs(pack_dir, exist_ok=True)
        LAST_WINPACK_STATE["pack_dir"] = pack_dir

    fname = f"{ev_key}.wav"
    fpath = os.path.join(pack_dir, fname)
    save_pcm16_wav(fpath, mastered_wav, sr)

    ev_map = dict(LAST_WINPACK_STATE.get("event_wav_map") or {})
    ev_map[ev_key] = {"path": fpath, "filename": fname, "prompt": ev_prompt}
    LAST_WINPACK_STATE["event_wav_map"] = ev_map

    s_name = (scheme_name or LAST_WINPACK_STATE.get("scheme_name") or "AEI_Custom_Scheme").strip()
    t_name = (theme_name or LAST_WINPACK_STATE.get("theme_name") or "Custom Theme").strip()
    m_lbl = (model_label or LAST_WINPACK_STATE.get("model_label") or "AEI Sound Studio").strip()

    zip_path = package_windows_sound_scheme_zip(
        scheme_name=s_name,
        event_wav_map=ev_map,
        theme_name=t_name,
        model_label=m_lbl,
        output_dir=output_dir,
    )
    LAST_WINPACK_STATE["pack_dir"] = pack_dir
    board_html = build_audition_soundboard_html(
        scheme_name=s_name,
        event_wav_map=ev_map,
        theme_name=t_name,
        model_label=m_lbl,
        embed_base64=True,
    )
    spec_img = render_waveform_and_spectrogram(fpath, title=f"Re-Rolled Event: {ev_key}")
    status = (
        f"🔄 Re-rolled **`{ev_key}`** ({WINDOWS_SYSTEM_EVENTS.get(ev_key, {}).get('label', ev_key)}) and updated "
        f"**`{os.path.basename(zip_path)}`** ({len(ev_map)} total events in pack)!"
    )
    return zip_path, fpath, spec_img, board_html, status


def discover_drive_videos():
    """Scan Google Drive and /content for MP4/MOV/WEBM videos from companion notebooks."""
    roots = ["/content/drive/MyDrive", "/content"]
    found = []
    for r in roots:
        if os.path.exists(r):
            for ext in ("*.mp4", "*.mov", "*.webm"):
                found.extend(glob.glob(os.path.join(r, "**", ext), recursive=True))
    found = sorted(set(found), key=lambda p: os.path.getmtime(p) if os.path.exists(p) else 0, reverse=True)
    return ["(Upload video below)"] + found[:40]


def image_to_temp_conditioning_video(image_path: str, duration_s: int = 10, fps: int = 25) -> str:
    """Convert a static desktop wallpaper/image into a 25-fps subtle Ken Burns conditioning MP4 for AudioX-Turbo & Synchformer."""
    img = Image.open(image_path).convert("RGB").resize((256, 256), Image.Resampling.LANCZOS)
    tmp_img = os.path.join("/tmp", f"wall_{uuid.uuid4().hex[:8]}.png")
    out_mp4 = os.path.join("/tmp", f"wall_cond_{uuid.uuid4().hex[:8]}.mp4")
    img.save(tmp_img)
    total_frames = int(duration_s * fps)
    vf_str = (
        f"zoompan=z='min(zoom+0.0008,1.12)':d={total_frames}:"
        f"x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':s=224x224:fps={int(fps)},format=yuv420p"
    )
    cmd = (
        f'ffmpeg -y -i "{tmp_img}" -t {int(duration_s)} '
        f'-vf "{vf_str}" -c:v libx264 -pix_fmt yuv420p -an "{out_mp4}"'
    )
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if not os.path.exists(out_mp4):
        cmd_fb = (
            f'ffmpeg -y -loop 1 -i "{tmp_img}" -t {int(duration_s)} '
            f'-r {int(fps)} -vf "scale=224:224,format=yuv420p" -c:v libx264 -pix_fmt yuv420p -an "{out_mp4}"'
        )
        subprocess.run(cmd_fb, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return out_mp4 if os.path.exists(out_mp4) else None


class AudioXTurboStudio:
    """Manager for HKUSTAudio/AudioX-Turbo (4-step DMD student + VAE + Synchformer)."""

    def __init__(self):
        self.model = None
        self.model_config = None
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.ckpt_dir = os.path.join(CACHE_ROOT, "checkpoints")
        os.makedirs(self.ckpt_dir, exist_ok=True)

    def unload(self):
        self.model = None
        self.model_config = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def ensure_checkpoints(self):
        files = [
            "audiox_turbo/audiox_turbo.ckpt",
            "pretransform/vae.ckpt",
            "synchformer/synchformer_state_dict.pth",
        ]
        paths = {}
        for rel in files:
            local_target = os.path.join(self.ckpt_dir, rel)
            if not os.path.exists(local_target):
                print(f"⬇️ Downloading {rel} from HKUSTAudio/AudioX-Turbo...")
                hf_hub_download(
                    repo_id="HKUSTAudio/AudioX-Turbo",
                    filename=rel,
                    local_dir=self.ckpt_dir,
                )
            paths[rel] = local_target
        repo_ckpt = os.path.join(REPO_DIR, "checkpoints")
        if not os.path.exists(repo_ckpt):
            try:
                os.symlink(self.ckpt_dir, repo_ckpt)
            except Exception:
                pass
        os.environ["AUDIOX_TURBO_SYNCHFORMER_CKPT"] = paths["synchformer/synchformer_state_dict.pth"]
        return paths

    def ensure_loaded(self):
        if self.model is not None and self.model_config is not None:
            return "✅ Using cached `AudioX-Turbo` (4-step DMD model)."
        paths = self.ensure_checkpoints()
        from audiox_turbo.inference import load_audiox_turbo_model
        cfg_path = os.path.join(REPO_DIR, "configs", "audiox_turbo_infer_4step.json")
        self.model, self.model_config = load_audiox_turbo_model(
            cfg_path,
            paths["audiox_turbo/audiox_turbo.ckpt"],
            pretransform_ckpt_path=paths["pretransform/vae.ckpt"],
            device=self.device,
        )
        return f"✅ Loaded `AudioX-Turbo` 4-step student on `{self.device}`."

    @torch.no_grad()
    def generate(
        self,
        prompt: str = "",
        video_path: str = None,
        audio_path: str = None,
        seconds_total: int = 10,
        steps: int = 4,
        seed: int = -1,
    ):
        msg = self.ensure_loaded()
        from audiox_turbo.inference.generation import generate_diffusion_cond_dmd
        from audiox_turbo.data.utils import (
            read_video,
            load_and_process_audio,
            encode_video_with_synchformer,
        )

        sample_rate = int(self.model_config.get("sample_rate", 44100))
        sample_size = int(self.model_config.get("sample_size", 441000))
        target_fps = int(self.model_config.get("video_fps", 5))
        model_sec = 10
        actual_seed = int(seed) if int(seed) >= 0 else int(np.random.randint(1, 999999))

        if video_path and os.path.exists(video_path):
            video_tensor = read_video(video_path, seek_time=0, duration=model_sec, target_fps=target_fps)
            sync_features = encode_video_with_synchformer(video_path, 0, model_sec, device=self.device)
        else:
            video_tensor = torch.zeros(model_sec * target_fps, 3, 224, 224)
            sync_features = torch.zeros(1, 240, 768, device=self.device)

        audio_tensor = load_and_process_audio(audio_path if (audio_path and os.path.exists(audio_path)) else None, sample_rate, 0, model_sec)

        conditioning = [{
            "video_prompt": {"video_tensors": video_tensor.unsqueeze(0), "video_sync_frames": sync_features},
            "text_prompt": prompt or "",
            "audio_prompt": audio_tensor.unsqueeze(0),
            "seconds_start": 0,
            "seconds_total": model_sec,
        }]

        audio = generate_diffusion_cond_dmd(
            self.model,
            steps=int(steps),
            conditioning=conditioning,
            sample_size=sample_size,
            seed=actual_seed,
            device=self.device,
        )
        req_samples = min(audio.shape[-1], int(float(seconds_total) * sample_rate))
        audio = audio[:, :, :req_samples]
        audio = rearrange(audio, "b d n -> n (b d)")
        arr = audio.detach().cpu().float().numpy()
        return arr, sample_rate, actual_seed, msg


AX_STUDIO = AudioXTurboStudio()
print("✅ Step 3 Complete: AudioXTurboStudio initialized!")

In [ ]:
#@title Step 4: Launch AudioX-Turbo Gradio Studio (4 Multimodal Tabs)
import os
import time
import subprocess
import gradio as gr
from IPython.display import clear_output


def ui_ax_text_to_audio(prompt, duration, steps, seed, apply_mastering, trim_db, fade_out_ms):
    try:
        if not prompt or not prompt.strip():
            return None, None, "❌ Please enter a text prompt."
        t0 = time.time()
        arr, sr, used_seed, load_msg = AX_STUDIO.generate(
            prompt=prompt.strip(), video_path=None, audio_path=None,
            seconds_total=float(duration), steps=int(steps), seed=int(seed),
        )
        if apply_mastering:
            arr = master_ui_sound(
                arr, sr, target_duration_s=float(duration), trim_silence=True,
                trim_db=float(trim_db), fade_in_ms=5.0, fade_out_ms=float(fade_out_ms),
            )
        out_wav = os.path.join(OUTPUT_DIR, f"audiox_turbo_t2a_{int(time.time())}_s{used_seed}.wav")
        save_pcm16_wav(out_wav, arr, sr)
        spec_img = render_waveform_and_spectrogram(out_wav, title=f"AudioX-Turbo ({steps}-Step DMD)")
        return out_wav, spec_img, f"✅ Generated in **{time.time()-t0:.2f}s** ({steps} steps, Seed `{used_seed}`).\n- {load_msg}"
    except Exception as e:
        return None, None, f"❌ AudioX-Turbo Error: {e}"


def ui_ax_wallpaper_windows_pack(
    wallpaper_img,
    scheme_name,
    theme_name,
    custom_style,
    selected_events,
    steps,
    base_seed,
    trim_db,
    fade_out_ms,
):
    try:
        if not selected_events:
            return None, None, None, "", "❌ Please select at least one Windows system event."
        t0 = time.time()
        cond_video = None
        if wallpaper_img and os.path.exists(wallpaper_img):
            cond_video = image_to_temp_conditioning_video(wallpaper_img, duration_s=10, fps=25)

        pack_dir = os.path.join(OUTPUT_DIR, f"ax_winpack_{int(time.time())}")
        os.makedirs(pack_dir, exist_ok=True)
        event_wav_map = {}
        first_wav = None
        seed_val = int(base_seed) if int(base_seed) >= 0 else int(np.random.randint(1000, 900000))

        for idx, ev_key in enumerate(selected_events):
            ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, WINDOWS_SYSTEM_EVENTS["SystemNotification"])
            ev_prompt = build_windows_event_prompt(ev_key, theme_name, custom_style)
            dur = float(ev_meta["duration"])
            arr, sr, _, _ = AX_STUDIO.generate(
                prompt=ev_prompt,
                video_path=cond_video,
                audio_path=None,
                seconds_total=max(dur + 0.5, 1.5),
                steps=int(steps),
                seed=seed_val + idx,
            )
            mastered = master_ui_sound(
                arr, sr, target_duration_s=dur, trim_silence=True,
                trim_db=float(trim_db), fade_in_ms=5.0, fade_out_ms=float(fade_out_ms),
            )
            fname = f"{ev_key}.wav"
            fpath = os.path.join(pack_dir, fname)
            save_pcm16_wav(fpath, mastered, sr)
            event_wav_map[ev_key] = {"path": fpath, "filename": fname, "prompt": ev_prompt}
            if first_wav is None:
                first_wav = fpath

        cond_tag = "Wallpaper+Text Conditioned" if cond_video else "Text Conditioned"
        model_lbl = f"AudioX-Turbo {steps}-Step ({cond_tag})"
        zip_path = package_windows_sound_scheme_zip(
            scheme_name=scheme_name,
            event_wav_map=event_wav_map,
            theme_name=f"{theme_name} ({cond_tag})",
            model_label=model_lbl,
            output_dir=OUTPUT_DIR,
        )
        board_html = build_audition_soundboard_html(
            scheme_name=scheme_name,
            event_wav_map=event_wav_map,
            theme_name=f"{theme_name} ({cond_tag})",
            model_label=model_lbl,
            embed_base64=True,
        )
        spec_img = render_waveform_and_spectrogram(first_wav, title=f"Event Preview: {selected_events[0]}") if first_wav else None
        status = (
            f"✅ Built **{cond_tag}** Windows Sound Pack `{os.path.basename(zip_path)}` "
            f"({len(event_wav_map)} events) in **{time.time()-t0:.1f}s**!"
        )
        return zip_path, first_wav, spec_img, board_html, status
    except Exception as e:
        return None, None, None, "", f"❌ Wallpaper Windows Pack Error: {e}"


def ui_ax_reroll_single_event(
    wallpaper_img,
    scheme_name,
    theme_name,
    custom_style,
    ev_key,
    ev_custom_prompt,
    reroll_seed,
    pitch_semitones,
    steps,
    trim_db,
    fade_out_ms,
):
    try:
        cond_video = None
        if wallpaper_img and os.path.exists(wallpaper_img):
            cond_video = image_to_temp_conditioning_video(wallpaper_img, duration_s=10, fps=25)
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, WINDOWS_SYSTEM_EVENTS["SystemNotification"])
        dur = float(ev_meta["duration"])
        ev_prompt = (ev_custom_prompt or "").strip() or build_windows_event_prompt(ev_key, theme_name, custom_style)
        arr, sr, _, _ = AX_STUDIO.generate(
            prompt=ev_prompt,
            video_path=cond_video,
            audio_path=None,
            seconds_total=max(dur + 0.5, 1.5),
            steps=int(steps),
            seed=int(reroll_seed),
        )
        mastered = master_ui_sound(
            arr, sr, target_duration_s=dur, trim_silence=True,
            trim_db=float(trim_db), fade_in_ms=1.5, fade_out_ms=float(fade_out_ms),
            pitch_semitones=float(pitch_semitones),
        )
        cond_tag = "Wallpaper+Text Conditioned" if cond_video else "Text Conditioned"
        return finalize_rerolled_event_in_pack(
            ev_key=ev_key,
            mastered_wav=mastered,
            sr=sr,
            ev_prompt=ev_prompt,
            output_dir=OUTPUT_DIR,
            scheme_name=scheme_name,
            theme_name=f"{theme_name} ({cond_tag})",
            model_label=f"AudioX-Turbo {steps}-Step ({cond_tag})",
        )
    except Exception as e:
        return None, None, None, "", f"❌ Single-Event Re-Roll Error: {e}"


def ui_ax_video_to_audio(drive_video, uploaded_video, prompt, duration, steps, seed):
    try:
        vid_path = uploaded_video if (uploaded_video and os.path.exists(uploaded_video)) else (
            drive_video if (drive_video and os.path.exists(drive_video)) else None
        )
        if not vid_path:
            return None, None, None, "❌ Please upload a video or select one from the Drive Video Picker."
        t0 = time.time()
        arr, sr, used_seed, _ = AX_STUDIO.generate(
            prompt=prompt.strip() or "Generate synchronized foley sound effects for the video",
            video_path=vid_path,
            audio_path=None,
            seconds_total=float(duration),
            steps=int(steps),
            seed=int(seed),
        )
        ts = int(time.time())
        out_wav = os.path.join(OUTPUT_DIR, f"audiox_v2a_{ts}_s{used_seed}.wav")
        out_mp4 = os.path.join(OUTPUT_DIR, f"audiox_v2a_{ts}_s{used_seed}.mp4")
        save_pcm16_wav(out_wav, arr, sr)

        cmd = (
            f'ffmpeg -y -i "{vid_path}" -i "{out_wav}" -t {float(duration)} '
            f'-c:v copy -map 0:v:0 -map 1:a:0 -shortest "{out_mp4}"'
        )
        subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        spec_img = render_waveform_and_spectrogram(out_wav, title="AudioX-Turbo V2A Foley")
        return out_mp4 if os.path.exists(out_mp4) else vid_path, out_wav, spec_img, (
            f"✅ Synchronized V2A Foley generated in **{time.time()-t0:.2f}s** (`{out_mp4}`)"
        )
    except Exception as e:
        return None, None, None, f"❌ Video-to-Audio Error: {e}"


def ui_ax_audio_conditioned(ref_audio, prompt, duration, steps, seed, apply_mastering):
    try:
        if not ref_audio or not os.path.exists(ref_audio):
            return None, None, "❌ Please upload a reference audio file."
        t0 = time.time()
        arr, sr, used_seed, _ = AX_STUDIO.generate(
            prompt=prompt.strip() or "High quality sound effect variation",
            video_path=None,
            audio_path=ref_audio,
            seconds_total=float(duration),
            steps=int(steps),
            seed=int(seed),
        )
        if apply_mastering:
            arr = master_ui_sound(arr, sr, target_duration_s=float(duration))
        out_wav = os.path.join(OUTPUT_DIR, f"audiox_a2a_{int(time.time())}_s{used_seed}.wav")
        save_pcm16_wav(out_wav, arr, sr)
        spec_img = render_waveform_and_spectrogram(out_wav, title="Audio-Conditioned Variation")
        return out_wav, spec_img, f"✅ Generated Audio+Text variation in **{time.time()-t0:.2f}s** (`{out_wav}`)"
    except Exception as e:
        return None, None, f"❌ Audio Conditioning Error: {e}"


with gr.Blocks(title="AudioX-Turbo — 4-Step Anything-to-Audio Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "## ⚡ AudioX-Turbo — 4-Step Anything-to-Audio, Wallpaper Sound Pack & Video Foley Studio\n"
        "Distilled via Distribution Matching Distillation (DMD) to synthesize **44.1 kHz stereo** audio in just **4 steps** "
        "from **Text, Desktop Wallpaper Images, Video, or Reference Audio**."
    )
    banner_md = gr.Markdown("⏳ Initializing AudioX-Turbo studio...")

    with gr.Tabs():
        with gr.Tab("⚡ 1. 4-Step Text-to-SFX & Music"):
            with gr.Row():
                with gr.Column(scale=3):
                    ax_preset = gr.Dropdown(
                        choices=[
                            "Crisp futuristic glass two-note ascending UI notification chime",
                            "Mechanical keyboard typing rapidly with tactile switch clicks",
                            "Heavy sci-fi spaceship airlock door hissing and locking shut",
                            "Warm ambient synth pad swell with crystalline bell overtones",
                            "Laser blaster burst followed by metallic ricochet echo",
                        ],
                        value="Crisp futuristic glass two-note ascending UI notification chime",
                        label="✨ Quick Prompt Preset",
                        info="Select a preset to populate the prompt box.",
                    )
                    ax_prompt = gr.Textbox(
                        value="Crisp futuristic glass two-note ascending UI notification chime",
                        lines=3,
                        label="📝 Text Prompt (SFX or Music)",
                        info="Describe the sound effect or musical stinger.",
                    )
                    with gr.Row():
                        ax_dur = gr.Slider(0.5, 10.0, value=3.0, step=0.5, label="⏱️ Duration (Seconds, up to 10s)", info="AudioX-Turbo operates on a 10s window and slices to your target duration.")
                        ax_steps = gr.Slider(1, 8, value=4, step=1, label="⚡ DMD Sampling Steps (Default: 4)", info="Distilled student is optimized for 4 steps (CFG-free).")
                        ax_seed = gr.Number(value=-1, precision=0, label="🎲 Seed (-1 = Random)", info="Random seed.")
                    with gr.Row():
                        ax_master = gr.Checkbox(value=True, label="✨ Zero-Latency UI Mastering", info="Auto-trim leading silence and apply micro-fades.")
                        ax_trim = gr.Slider(-60.0, -25.0, value=-42.0, step=1.0, label="✂️ Trim Threshold (dB)", info="Silence threshold.")
                        ax_fade = gr.Slider(20.0, 350.0, value=120.0, step=10.0, label="📉 Fade-Out (ms)", info="Smooth tail fade.")
                    ax_btn = gr.Button("⚡ Generate in 4 Steps", variant="primary")
                with gr.Column(scale=2):
                    ax_wav = gr.Audio(label="🎧 44.1 kHz Stereo Output", type="filepath")
                    ax_spec = gr.Image(label="📊 Waveform & Spectrogram", type="filepath")
                    ax_status = gr.Markdown()

            ax_preset.change(fn=lambda x: x, inputs=[ax_preset], outputs=[ax_prompt])
            ax_btn.click(
                fn=ui_ax_text_to_audio,
                inputs=[ax_prompt, ax_dur, ax_steps, ax_seed, ax_master, ax_trim, ax_fade],
                outputs=[ax_wav, ax_spec, ax_status],
            )

        with gr.Tab("🖼️ 2. Wallpaper + Text → Windows Sound Pack"):
            gr.Markdown(
                "### 🖼️ Desktop Wallpaper + Text $\\rightarrow$ Visually Aligned Windows Sound Event Pack\n"
                "Upload your **desktop wallpaper** (optional)! `AudioX-Turbo`'s Multimodal Adaptive Fusion (MAF) conditions "
                "on both the visual aesthetic of your wallpaper (`CLIP` + `Synchformer`) and each Windows event description."
            )
            with gr.Row():
                with gr.Column(scale=3):
                    wp_img = gr.Image(label="🖼️ Upload Desktop Wallpaper Image (Optional Visual Conditioning)", type="filepath", height=200)
                    with gr.Row():
                        wp_scheme = gr.Textbox(value="AudioX_Wallpaper_Theme", label="🏷️ Windows Sound Scheme Name", info="Display name in Windows Sound Control Panel.")
                        wp_theme = gr.Dropdown(
                            choices=list(SONIC_THEME_PRESETS.keys()),
                            value=list(SONIC_THEME_PRESETS.keys())[0],
                            label="🎨 Sonic Theme Preset",
                            info="Base sonic palette combined with your wallpaper conditioning.",
                        )
                    wp_custom = gr.Textbox(value="", label="✍️ Custom Theme Override (Optional)", info="Leave blank to use preset.")
                    wp_events = gr.CheckboxGroup(
                        choices=list(WINDOWS_SYSTEM_EVENTS.keys()),
                        value=DEFAULT_SELECTED_WIN_EVENTS,
                        label="🔔 Windows System Events to Generate (4 Steps Each!)",
                        info="Because AudioX-Turbo takes only 4 steps per event, a 10-event pack finishes in seconds!",
                    )
                    with gr.Row():
                        wp_steps = gr.Slider(2, 8, value=4, step=1, label="⚡ Steps per Event", info="Default: 4 steps.")
                        wp_seed = gr.Number(value=42, precision=0, label="🎲 Base Seed", info="Seed.")
                        wp_trim = gr.Slider(-60.0, -25.0, value=-42.0, step=1.0, label="✂️ Trim (dB)", info="Zero-latency trim.")
                        wp_fade = gr.Slider(40.0, 300.0, value=120.0, step=10.0, label="📉 Fade-Out (ms)", info="Release fade.")
                    wp_btn = gr.Button("🪟 Build Wallpaper-Aligned Windows Sound Pack (.zip + .ps1 + .reg)", variant="primary")
                    with gr.Accordion("🎯 Single-Event Re-Roll & Pitch Customizer (Update 1 Event Without Regenerating Pack)", open=False):
                        with gr.Row():
                            wp_rr_ev = gr.Dropdown(choices=list(WINDOWS_SYSTEM_EVENTS.keys()), value="DeviceConnect", label="🔔 Event to Re-Roll / Replace", info="Pick a single Windows event to re-roll in 4 steps.")
                            wp_rr_seed = gr.Number(value=-1, precision=0, label="🎲 New Seed (-1 = Random)", info="Fresh seed.")
                            wp_rr_pitch = gr.Slider(-6.0, 6.0, value=0.0, step=0.5, label="🎵 Pitch Transpose (Semitones)", info="Semitone shift.")
                        wp_rr_prompt = gr.Textbox(value="", placeholder="Optional custom prompt override for this event...", label="✍️ Single-Event Prompt Override (Optional)", info="Leave blank to use theme prompt.")
                        wp_rr_btn = gr.Button("🔄 Re-Roll Single Event in 4 Steps & Update Pack")
                with gr.Column(scale=2):
                    wp_zip = gr.File(label="📦 Download Windows Sound Scheme (.zip)")
                    wp_prev = gr.Audio(label="🎧 Sample Event Preview", type="filepath")
                    wp_spec = gr.Image(label="📊 Spectrogram Preview", type="filepath")
                    wp_status = gr.Markdown()
            wp_board = gr.HTML(label="🎹 Interactive Wallpaper Windows Sound Scheme Audition Board")

            wp_btn.click(
                fn=ui_ax_wallpaper_windows_pack,
                inputs=[wp_img, wp_scheme, wp_theme, wp_custom, wp_events, wp_steps, wp_seed, wp_trim, wp_fade],
                outputs=[wp_zip, wp_prev, wp_spec, wp_board, wp_status],
            )
            wp_rr_btn.click(
                fn=ui_ax_reroll_single_event,
                inputs=[wp_img, wp_scheme, wp_theme, wp_custom, wp_rr_ev, wp_rr_prompt, wp_rr_seed, wp_rr_pitch, wp_steps, wp_trim, wp_fade],
                outputs=[wp_zip, wp_prev, wp_spec, wp_board, wp_status],
            )

        with gr.Tab("🎬 3. 4-Step Video-to-Audio (V2A) Foley"):
            with gr.Row():
                with gr.Column():
                    with gr.Row():
                        v_drive = gr.Dropdown(
                            choices=discover_drive_videos(),
                            value="(Upload video below)",
                            label="📂 Cross-Notebook Google Drive Video Picker",
                            info="Pick any video exported from Wan2.2, Wan-VACE, FramePack, LTX-2, or SeedVR2.",
                            scale=3,
                        )
                        v_ref_btn = gr.Button("🔄 Refresh Drive", scale=1)
                    v_upload = gr.Video(label="🎬 Or Upload Video (.mp4 / .mov)")
                    v_prompt = gr.Textbox(
                        value="Generate realistic synchronized foley sound effects matching the actions in the video",
                        label="📝 V2A / TV2A Prompt",
                        info="Use 'Generate general audio for the video' or describe specific foley/music.",
                    )
                    with gr.Row():
                        v_dur = gr.Slider(1.0, 10.0, value=8.0, step=0.5, label="⏱️ Duration (s, up to 10s)", info="Synchronized window length.")
                        v_steps = gr.Slider(2, 8, value=4, step=1, label="⚡ DMD Steps", info="Default: 4.")
                        v_seed = gr.Number(value=-1, precision=0, label="🎲 Seed", info="Seed.")
                    v_btn = gr.Button("🎬 Generate 4-Step Synchronized Foley", variant="primary")
                with gr.Column():
                    v_out_mp4 = gr.Video(label="🎥 Video with Synchronized AudioX-Turbo Foley")
                    v_out_wav = gr.Audio(label="🎧 Isolated Stereo Foley Track (.wav)", type="filepath")
                    v_spec = gr.Image(label="📊 Spectrogram", type="filepath")
                    v_status = gr.Markdown()

            v_ref_btn.click(fn=lambda: gr.update(choices=discover_drive_videos()), inputs=None, outputs=[v_drive])
            v_btn.click(
                fn=ui_ax_video_to_audio,
                inputs=[v_drive, v_upload, v_prompt, v_dur, v_steps, v_seed],
                outputs=[v_out_mp4, v_out_wav, v_spec, v_status],
            )

        with gr.Tab("🔄 4. Audio + Text Conditioning (TA2A)"):
            with gr.Row():
                with gr.Column():
                    a_ref = gr.Audio(label="📂 Upload Reference Audio (.wav / .mp3)", type="filepath")
                    a_prompt = gr.Textbox(value="Futuristic sci-fi synthesizer chime variation", label="📝 Target Prompt", info="Steer the audio prompt toward a new texture.")
                    with gr.Row():
                        a_dur = gr.Slider(0.5, 10.0, value=3.0, step=0.5, label="⏱️ Duration (s)", info="Output length.")
                        a_steps = gr.Slider(2, 8, value=4, step=1, label="⚡ Steps", info="Default 4.")
                        a_seed = gr.Number(value=-1, precision=0, label="🎲 Seed", info="Seed.")
                    a_master = gr.Checkbox(value=True, label="✨ Apply Zero-Latency UI Mastering", info="Auto-trim leading silence.")
                    a_btn = gr.Button("🔄 Generate Audio-Conditioned Variation", variant="primary")
                with gr.Column():
                    a_out = gr.Audio(label="🎧 Output Audio", type="filepath")
                    a_spec = gr.Image(label="📊 Spectrogram", type="filepath")
                    a_status = gr.Markdown()

            a_btn.click(
                fn=ui_ax_audio_conditioned,
                inputs=[a_ref, a_prompt, a_dur, a_steps, a_seed, a_master],
                outputs=[a_out, a_spec, a_status],
            )

    demo.load(
        fn=lambda: f"🟢 **AudioX-Turbo (4-Step DMD Anything-to-Audio) Ready** on `{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}`!",
        inputs=None,
        outputs=[banner_md],
    )

clear_output()
demo.queue(default_concurrency_limit=1)
demo.launch(share=True, inline=False, show_error=True)

In [ ]:
#@title Step 5: Keep-Alive Helper & GPU VRAM Monitor
import time
import torch

KEEP_ALIVE_MINUTES = 0 #@param {type:"integer"}

try:
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated(0) / (1024 ** 3)
        resv = torch.cuda.memory_reserved(0) / (1024 ** 3)
        print(f"📊 GPU VRAM Status — Allocated: {alloc:.2f} GB | Reserved: {resv:.2f} GB")
    if KEEP_ALIVE_MINUTES > 0:
        print(f"⏳ Keeping Gradio session alive for {KEEP_ALIVE_MINUTES} minutes...")
        for m in range(KEEP_ALIVE_MINUTES):
            time.sleep(60)
            print(f"  ✓ Active: {m + 1}/{KEEP_ALIVE_MINUTES} min elapsed")
    else:
        print("ℹ️ Keep-alive loop skipped (KEEP_ALIVE_MINUTES=0). Set > 0 if running unattended.")
except KeyboardInterrupt:
    print("⏹️ Keep-alive stopped by user.")
except Exception as e:
    print(f"Keep-alive warning: {e}")

In [ ]:
#@title Step 6: Quick Test — 4-Step Stereo SFX Synthesis
import os
from IPython.display import Audio, FileLink, display

QUICKTEST_PROMPT = "Crisp futuristic glass two-note ascending UI notification chime" #@param {type:"string"}
QUICKTEST_DURATION = 2.5 #@param {type:"number"}
QUICKTEST_SEED = 42 #@param {type:"integer"}

try:
    wav_path, _, msg = ui_ax_text_to_audio(
        prompt=QUICKTEST_PROMPT,
        duration=QUICKTEST_DURATION,
        steps=4,
        seed=QUICKTEST_SEED,
        apply_mastering=True,
        trim_db=-42.0,
        fade_out_ms=120.0,
    )
    print(msg)
    if wav_path and os.path.exists(wav_path):
        display(Audio(filename=wav_path))
        display(FileLink(wav_path))
except Exception as e:
    print(f"Quick test error: {e}")

In [ ]:
#@title Step 7: Headless Batch 4-Step Windows Sound Scheme Generator
import os
from tqdm.auto import tqdm
from IPython.display import FileLink, display

BATCH_SCHEME_NAME = "AudioX_Turbo_4Step" #@param {type:"string"}
BATCH_THEME = "Cyberpunk Neon HUD" #@param ["Frutiger Aero Crystal Glass", "Cyberpunk Neon HUD", "Cozy Wooden Kalimba & Mallet", "Sci-Fi Spaceship Bridge", "Retro 16-Bit Chiptune Arcade", "Warm Lo-Fi Analog Tape Synth", "Minimalist Tactile Micro-Foley", "Dark Fantasy Crystal & Harp"]
BATCH_SEED = 42 #@param {type:"integer"}

try:
    event_map = {}
    pack_dir = os.path.join(OUTPUT_DIR, f"batch_ax_winpack_{BATCH_SEED}")
    os.makedirs(pack_dir, exist_ok=True)
    for idx, ev_key in enumerate(tqdm(DEFAULT_SELECTED_WIN_EVENTS, desc="AudioX-Turbo 4-Step Windows Pack")):
        try:
            ev_meta = WINDOWS_SYSTEM_EVENTS[ev_key]
            prompt = build_windows_event_prompt(ev_key, BATCH_THEME, "")
            arr, sr, _, _ = AX_STUDIO.generate(
                prompt=prompt,
                seconds_total=max(float(ev_meta["duration"]) + 0.5, 1.5),
                steps=4,
                seed=BATCH_SEED + idx,
            )
            mastered = master_ui_sound(arr, sr, target_duration_s=float(ev_meta["duration"]))
            fpath = os.path.join(pack_dir, f"{ev_key}.wav")
            save_pcm16_wav(fpath, mastered, sr)
            event_map[ev_key] = {"path": fpath, "filename": f"{ev_key}.wav", "prompt": prompt}
        except Exception as item_e:
            print(f"⚠️ Event {ev_key} skipped: {item_e}")

    if event_map:
        zip_out = package_windows_sound_scheme_zip(BATCH_SCHEME_NAME, event_map, BATCH_THEME, "AudioX-Turbo (4-Step DMD)", OUTPUT_DIR)
        print(f"✅ Packaged Windows Sound Scheme: {zip_out}")
        display(FileLink(zip_out))
except Exception as e:
    print(f"Batch Windows Pack error: {e}")